# 01 — EDA + Brand Selection

Exploratory analysis of the Kaggle Customer Support on Twitter dataset.

**Goal**: pick one brand and justify the choice.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load raw CSV (subsample for speed)
df = pd.read_csv(
    "../data/raw/twcs.csv",
    usecols=["tweet_id", "author_id", "inbound", "text", "created_at"],
    dtype={"tweet_id": str},
    nrows=200_000,
    low_memory=False,
)
print(f"Loaded {len(df):,} rows")
df.head()


In [ ]:
# Top brand accounts by tweet volume (outbound = brand replies)
brand_counts = (
    df[df["inbound"] == False]["author_id"]
    .value_counts()
    .head(20)
)
print(brand_counts.to_string())


In [ ]:
# Plot top brands
fig, ax = plt.subplots(figsize=(12, 5))
brand_counts.plot(kind="bar", ax=ax, color="#1DB954")
ax.set_title("Top 20 Brand Accounts by Tweet Volume")
ax.set_xlabel("Brand")
ax.set_ylabel("Number of support replies")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../report/brand_volume.png", dpi=120)
plt.show()


## Brand Selection: SpotifyCare

**Chosen brand**: `@SpotifyCares`

**Rationale**:
1. **Narrow, well-defined domain** — music streaming with ~9 clearly separable intents
2. **Distinctive brand voice** — empathetic, concise, emoji-light, action-oriented
3. **Sufficient volume** — enough tweets for stratified golden-set sampling
4. **Not over-used** — Apple Support and Amazon Help are the most common choices in assignments; SpotifyCare gives a fresher angle with equally good data quality
5. **No order-tracking dependency** — unlike Amazon, Spotify issues are self-contained (no external fulfilment data needed)

**Brands considered but not chosen**:
- `AppleSupport` — high volume but 20+ product lines; intent space is too wide for a one-week scope
- `AmazonHelp` — high volume but order-specific issues require account context the agent can't access
- `XboxSupport` — narrower but smaller dataset; fewer examples per intent


In [ ]:
# SpotifyCares conversation volume over time
spotify_df = df[df["author_id"] == "SpotifyCares"].copy()
spotify_df["created_at"] = pd.to_datetime(spotify_df["created_at"], errors="coerce")
spotify_df = spotify_df.dropna(subset=["created_at"])
spotify_df["month"] = spotify_df["created_at"].dt.to_period("M")

monthly = spotify_df.groupby("month").size()
monthly.index = monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(12, 4))
monthly.plot(ax=ax, color="#1DB954", linewidth=2)
ax.set_title("SpotifyCare — Monthly Tweet Volume")
ax.set_xlabel("Month")
ax.set_ylabel("Tweets")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Total SpotifyCare tweets in sample: {len(spotify_df):,}")


In [ ]:
# Message length distribution for SpotifyCare customer messages
# (need to join with customer messages that SpotifyCare replied to)
spotify_replies = df[df["author_id"] == "SpotifyCares"].copy()
print(f"SpotifyCare reply stats:")
print(f"  Count: {len(spotify_replies):,}")
print(f"  Mean reply length: {spotify_replies['text'].str.len().mean():.0f} chars")
print(f"  Median reply length: {spotify_replies['text'].str.len().median():.0f} chars")
